<h1>One-Hot-Encoding</h1>

<p>Dieses Skript tut alle Spalten, die weniger als 200 verschiedene Werte haben, One-Hot-Encoden. Spalten mit mehr verschiedenen Werten sind Freitextfelder und werden anders behandelt.</p>

In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer


In [ ]:
df = pd.read_csv("Abgaben/survey_results_cleaned_final.csv")
max_unique = 200
na_as_category = True
TOP_N = 10


In [ ]:
df.LanguageHaveWorkedWith

In [ ]:
multiselect_cols = [
    "LanguageHaveWorkedWith",
    "DatabaseHaveWorkedWith",
    "PlatformHaveWorkedWith",
    "WebframeHaveWorkedWith",
    "DevEnvsHaveWorkedWith",
    "OfficeStackAsyncHaveWorkedWith",
    "AIModelsHaveWorkedWith",
]
multiselect_cols = [c for c in multiselect_cols if c in df.columns]

SEP = ";"

def split_cell(x):
    # Missing/leer -> entweder [] oder ["none"]
    if pd.isna(x) or str(x).strip() == "":
        return ["none"] if na_as_category else []
    return [p.strip() for p in str(x).split(SEP) if p.strip()]

def multiselect_topN_with_other(df: pd.DataFrame, col: str, top_n: int = 10, na_as_category: bool = True):
    """
    Multi-Select (z.B. 'java;python;sql') -> Multi-Hot Encoding
    Behalte global Top-N Kategorien, bündle Rest in 'other'.
    Missing/leer -> 'none' (wenn na_as_category=True)
    """
    s = df[col].apply(split_cell)

    # Top-N global bestimmen (ohne 'none')
    exploded = s.explode()
    counts = exploded[exploded != "none"].value_counts()
    top = set(counts.head(top_n).index)

    def keep_top_and_other(items):
        if na_as_category and items == ["none"]:
            return ["none"]

        kept = [x for x in items if x in top]
        has_other = any((x not in top) and (x != "none") for x in items)

        out = set(kept)
        if has_other:
            out.add("other")
        if na_as_category and ("none" in items):
            out.add("none")
        return sorted(out)

    s2 = s.apply(keep_top_and_other)

    classes = sorted(top) + ["other"]
    if na_as_category:
        classes = ["none"] + classes

    mlb = MultiLabelBinarizer(classes=classes)
    dummies = pd.DataFrame(mlb.fit_transform(s2), columns=mlb.classes_, index=df.index)

    # Prefix wie vorher mit Spaltennamen
    dummies = dummies.add_prefix(f"{col}__")

    return dummies, sorted(top)

# --- Multi-Select: Top10 + other ---
for col in multiselect_cols:
    dummies, top10 = multiselect_topN_with_other(df, col, top_n=TOP_N, na_as_category=na_as_category)
    df = df.drop(columns=[col]).join(dummies)

    print(f"Top{TOP_N} für {col}: {top10}")

print("Form nach Multi-Select-OHE (Top10+other):", df.shape)

In [59]:
df

,ResponseId,MainBranch,Age,MaxAge,AgeNum,EdLevel,Employment,WorkExp,LearnCodeAI,YearsCode,...,AIModelsHaveWorkedWith__deepseek (r- reasoning models),AIModelsHaveWorkedWith__deepseek (v- general purpose models),AIModelsHaveWorkedWith__gemini (flash general purpose models),AIModelsHaveWorkedWith__gemini (pro reasoning models),AIModelsHaveWorkedWith__meta llama (all models),AIModelsHaveWorkedWith__openai gpt (chatbot models),AIModelsHaveWorkedWith__openai image generating models,AIModelsHaveWorkedWith__openai reasoning models,AIModelsHaveWorkedWith__x grok models,AIModelsHaveWorkedWith__other
0,1,i am a developer by profession,25-34 years old,34.0,29.0,master’s degree,employed,8.0,"yes, i learned how to use ai-enabled tools for...",14.0,...,0,0,0,0,0,1,1,1,0,0
1,2,i am a developer by profession,25-34 years old,34.0,29.0,associate degree,employed,2.0,"yes, i learned how to use ai-enabled tools for...",10.0,...,0,0,0,0,0,1,0,0,0,0
2,3,i am a developer by profession,35-44 years old,44.0,39.0,bachelor’s degree,"independent contractor, freelancer, or self-em...",10.0,"yes, i learned how to use ai-enabled tools for...",12.0,...,0,0,1,0,0,1,0,0,0,0
3,4,i am a developer by profession,35-44 years old,44.0,39.0,bachelor’s degree,employed,4.0,"yes, i learned how to use ai-enabled tools for...",5.0,...,0,0,0,0,0,0,0,0,0,0
4,5,i am a developer by profession,35-44 years old,44.0,39.0,master’s degree,"independent contractor, freelancer, or self-em...",21.0,"yes, i learned how to use ai-enabled tools for...",22.0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20598,49011,i am a developer by profession,25-34 years old,34.0,29.0,bachelor’s degree,employed,4.0,"yes, i learned how to use ai-enabled tools req...",6.0,...,0,0,0,0,0,1,0,1,0,0
20599,49018,i am a developer by profession,18-24 years old,24.0,21.0,master’s degree,"independent contractor, freelancer, or self-em...",5.0,"yes, i learned how to use ai-enabled tools for...",6.0,...,0,0,0,0,0,0,0,0,0,0
20600,49067,i am a developer by profession,35-44 years old,44.0,39.0,bachelor’s degree,employed,19.0,"yes, i learned how to use ai-enabled tools req...",22.0,...,0,0,0,0,0,0,0,1,0,0
20601,49075,i am a developer by profession,25-34 years old,34.0,29.0,master’s degree,employed,5.0,"yes, i learned how to use ai-enabled tools for...",6.0,...,1,0,0,0,0,1,0,0,0,0


In [ ]:


print("Ursprüngliche DataFrame-Form:", df.shape)


In [ ]:
object_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Object-Spalten:")
object_columns


In [ ]:
if object_columns:
    nunique_per_column = df[object_columns].nunique(dropna=True)
else:
    nunique_per_column = pd.Series(dtype=int)

nunique_per_column


In [ ]:
columns_to_encode = nunique_per_column[nunique_per_column <= max_unique].index.tolist()

#exclude_columns = {'xxx'}

#columns_to_encode = [
#    col for col in columns_to_encode
#    if col not in exclude_columns
#]

high_cardinality_columns = nunique_per_column[nunique_per_column > max_unique].index.tolist()

print("Spalten für One-Hot-Encoding (≤ 50 Werte):")
print(columns_to_encode)

print("\nSpalten mit hoher Kardinalität (> 50 Werte):")
high_cardinality_columns


In [ ]:
df_encoded = pd.get_dummies(
    df,
    columns=columns_to_encode,
    dummy_na=na_as_category,
    drop_first=False
)

print("Neue DataFrame-Form nach One-Hot-Encoding:", df_encoded.shape)


In [ ]:
if high_cardinality_columns:
    print("Nicht encodierte Spalten mit mehr als", max_unique, "verschiedenen Einträgen:")
    for col in high_cardinality_columns:
        print(f" - {col}: {int(nunique_per_column[col])} unique values")
else:
    print("Keine Spalten mit hoher Kardinalität gefunden.")


In [ ]:
df_encoded.to_csv("One-Hot-Encoded.csv", index=False)

In [ ]:
df_encoded.shape

In [61]:
# multiselect_cols:
#     "LanguageHaveWorkedWith",
#     "DatabaseHaveWorkedWith",
#     "PlatformHaveWorkedWith",
#     "WebframeHaveWorkedWith",
#     "DevEnvsHaveWorkedWith",
#     "OfficeStackAsyncHaveWorkedWith",
#     "AIModelsHaveWorkedWith",

lang_cols = [c for c in df_encoded.columns if c.startswith("PlatformChoice")]
print(f"Gefunden: {len(lang_cols)} Spalten")
print(*lang_cols, sep="\n")

Gefunden: 3 Spalten
PlatformChoice_no
PlatformChoice_yes
PlatformChoice_nan
